# LegalIR Task 1: Google Colab A100 Production Training (B1.2)
## UIT Data Science Challenge 2026 — High-Recall Vietnamese Legal IR
**Pinned Git Commit:** `6b57ed7bfef94f8daaa823546c19d761b68b8755`

### Production Training Invariants:
- **Enforces NVIDIA A100 GPU** before training (a GPU runtime is already allocated when that cell executes).
- Verifies prior Kaggle Dual-T4 report (sole pre-A100 hardware gate).
- Trains `BAAI/bge-reranker-v2-m3` LoRA on all 7,000 canonical training queries.
- Uses `torch.bfloat16` precision end-to-end.
- Generates Top-5 predictions for 1,000 official public test queries.
- Verifies all submission invariants and builds `submission.zip`.
- Captures immutable Hugging Face release revision into `run_manifest.json`.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Verification & Local Environment Loader
# Fail-closed A100 enforcement BEFORE training (a GPU runtime is
# already allocated when this cell executes).
# ==============================================================================
import json
import os
import sys
import torch
from pathlib import Path

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU required for training."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[+] Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
assert "A100" in gpu_name, f"A100 required (found {gpu_name}). Select Runtime -> Change runtime type -> A100."

# Prefer Colab Secrets, fallback to uploaded .env (CLI automation)
# Precedence: Colab Secrets win over uploaded .env (Secrets set first,
# .env uses setdefault so it never overwrites Secrets); uploaded .env
# itself is filtered to an allowlist by scripts/colab/run_colab_cli.sh.
try:
    from google.colab import userdata
    _ud = userdata.get
except Exception:
    _ud = None
if _ud is not None:
    for _k in ["HF_TOKEN", "HF_TOKEN_WRITE", "HF_TOKEN_READ", "KAGGLE_API_TOKEN", "KAGGLE_KEY", "KAGGLE_USERNAME", "HF_REPO_ID", "HF_ALLOW_PUBLIC_REPO", "LEGALIR_COMMIT_SHA"]:
        try:
            _v = _ud(_k)
        except Exception:
            _v = None
        if _v and _k not in os.environ:
            os.environ[_k] = str(_v)

for env_path in [Path("/content/.env"), Path("/content/LegalIR/.env"), Path(".env")]:
    if env_path.is_file():
        print(f"[+] Loading local environment from {env_path}...")
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip("'\""))

# Normalize HF Token (fine-grained or classic; accept HF_TOKEN_WRITE, HF_TOKEN, HF_TOKEN_READ; ignore non-HF tokens like KGAT_)
# Required scopes for release: read access to public models + write access to HF_REPO_ID.
hf_candidates = [os.environ.get("HF_TOKEN_WRITE"), os.environ.get("HF_TOKEN"), os.environ.get("HF_TOKEN_READ")]
hf_tok = next((t for t in hf_candidates if t and str(t).startswith("hf_")), None)
if not hf_tok:
    raise RuntimeError("HF_TOKEN_WRITE or HF_TOKEN is required for model/log/submission delivery.")
os.environ["HF_TOKEN"] = hf_tok
# Verify write access after dependencies are installed; never print token values.


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Detached HEAD Checkout
# ==============================================================================
import subprocess
from pathlib import Path

APPROVED_COMMIT = "6b57ed7bfef94f8daaa823546c19d761b68b8755"
import re
launch_path = Path("/content/legalir_launch.json")
launch = json.loads(launch_path.read_text()) if launch_path.is_file() else {}
# Release selection precedence: CLI launch JSON first, then an explicit
# LEGALIR_COMMIT_SHA (manual Colab Secrets/env). The baked-in pin below
# is the RUNTIME build, not a release approval, so falling back to it
# silently would check out stale evidence and fail strict bootstrap only
# after GPU allocation and dependency installation. Manual runs must
# therefore select the evidence-bearing RELEASE commit explicitly; the
# CLI wrapper always supplies launch JSON. (A release's own notebook
# pins its runtime by design — embedding the release SHA here would be
# self-referential — so the operator provides it at launch.)
launch_expected = launch.get("expected_sha")
env_expected = os.environ.get("LEGALIR_COMMIT_SHA")
if launch_expected:
    EXPECTED_COMMIT = launch_expected
elif env_expected:
    EXPECTED_COMMIT = env_expected
else:
    raise RuntimeError(
        "No explicit release selection: set LEGALIR_COMMIT_SHA to the evidence-bearing "
        "release commit (the commit that passed strict release verification), or launch "
        "via scripts/colab/run_colab_cli.sh which supplies /content/legalir_launch.json."
    )
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_COMMIT):
    raise RuntimeError("LEGALIR_COMMIT_SHA must be an exact 40-character lowercase Git SHA.")
REPO_DIR = Path("/content/LegalIR") if Path("/content").exists() else Path.cwd()
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
actual = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip()
if actual != EXPECTED_COMMIT:
    raise RuntimeError("Checked-out runtime does not match the selected SHA.")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if not (REPO_DIR / "scripts/colab/bootstrap.py").is_file():
    raise RuntimeError("Selected approved runtime predates this launcher. Set LEGALIR_COMMIT_SHA to a published runtime with matching real T4 reports and freeze; old reports cannot approve new code.")
print(f"[+] Working in: {REPO_DIR} at {actual}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies & Canonical Dataset Setup
# Do NOT reinstall torch (Colab CUDA build). Install only missing wheels.
# ==============================================================================
import subprocess
import sys
from pathlib import Path

# A resolver conflict must fail rather than replace the loaded CUDA torch build.
constraints = Path("/content/legalir-torch-constraint.txt") if Path("/content").exists() else REPO_DIR / "artifacts/colab-torch-constraint.txt"
constraints.parent.mkdir(parents=True, exist_ok=True)
constraints.write_text(f"torch=={torch.__version__}\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints), "-r", str(REPO_DIR / "requirements-colab.txt")], check=True)
from scripts.colab.bootstrap import prepare_dataset, verify_launch
k_report_p = next(p for p in [Path("/content/kaggle_t4x2_report.json"), REPO_DIR / "artifacts/task1/gates/kaggle_t4x2_report.json"] if p.is_file())
freeze_p = next(p for p in [Path("/content/production_freeze.json"), REPO_DIR / "artifacts/task1/freeze/production_freeze.json"] if p.is_file())
verify_launch(EXPECTED_COMMIT, k_report_p, freeze_p)
from scripts.gates.run_a100 import preflight_huggingface_access
hf_repo = os.environ.get("HF_REPO_ID", "dangphuc2109/legalir-task1-reranker")
# Explicit opt-in only: absent or "0" means private-only; literal "1" permits
# pushing to an existing PUBLIC repo (recorded in manifest). Arbitrary
# nonempty strings do NOT count as consent.
hf_allow_public = os.environ.get("HF_ALLOW_PUBLIC_REPO", "0") == "1"
print("[!] OPERATOR OVERRIDE: public HF repos permitted for this launch.") if hf_allow_public else None
hf_ok, hf_detail = preflight_huggingface_access(hf_repo, allow_public_repo=hf_allow_public)
if not hf_ok:
    raise RuntimeError(hf_detail)
print(f"[+] {hf_detail}")
dataset_dir = Path("/content/kaggle_dataset") if Path("/content").exists() else REPO_DIR / "artifacts/task1/data"
prepare_dataset(dataset_dir, freeze_p)
print(f"[+] Dataset fingerprint verified at: {dataset_dir}")


In [ ]:
# ==============================================================================
# Cell 4: Execute Colab A100 Production Gate (scripts/run_colab_train.py -> scripts/gates/run_a100.py)
# CLI equivalent: python scripts/run_colab_train.py --dataset-dir /content/kaggle_dataset --output-dir /content/legalir_production_run --expected-sha $EXPECTED_COMMIT
# ==============================================================================
from scripts.run_colab_train import run_colab_production_training

output_dir = Path("/content/legalir_production_run") if Path("/content").exists() else REPO_DIR / "artifacts/task1/production"
output_dir.mkdir(parents=True, exist_ok=True)

hf_repo = os.environ.get("HF_REPO_ID", "dangphuc2109/legalir-task1-reranker")
k_cands = [Path("/content/kaggle_t4x2_report.json"), REPO_DIR / "artifacts/task1/gates/kaggle_t4x2_report.json"]
k_report_p = next((p for p in k_cands if p.is_file()), None)
freeze_cands = [Path("/content/production_freeze.json"), REPO_DIR / "artifacts/task1/freeze/production_freeze.json"]
freeze_p = next((p for p in freeze_cands if p.is_file()), None)

report = run_colab_production_training(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    smoke_report_path=k_report_p,
    expected_sha=EXPECTED_COMMIT,
    precision="bf16",
    allow_non_a100=False,
    mock=False,
    hf_repo=hf_repo,
    freeze_file_path=freeze_p,
    run_mode="full",
    hf_allow_public_repo=hf_allow_public,
)
print(f"[+] A100 Production Gate execution status: {report.get('status')} | Verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Verify Submission & Report Release State
# ==============================================================================
from src.evaluation.submission import validate_submission_zip

sub_zip = output_dir / "submission.zip"
zip_val = validate_submission_zip(sub_zip)
assert zip_val.get("is_valid"), f"Submission validation failed: {zip_val.get('errors')}"
print(f"[+] SUCCESS: submission.zip validated cleanly at {sub_zip}")

manifest_p = output_dir / "run_manifest.json"
assert manifest_p.is_file(), f"Run manifest missing at {manifest_p}"
m_data = json.loads(manifest_p.read_text(encoding='utf-8'))
print(f"[+] Run Status : {m_data.get('status')}")
print(f"[+] Verdict    : {m_data.get('verdict')}")
hf_meta = m_data.get("huggingface", {})
assert m_data.get("status") == "RELEASED" and hf_meta.get("uploaded"), "Training completed but Hugging Face delivery did not. Recover local artifacts."
assert hf_meta.get("manifest_commit_sha"), "Missing immutable release receipt."
print(f"[+] Hugging Face release: https://huggingface.co/{hf_meta['repo_id']}/tree/{hf_meta['manifest_commit_sha']}/{hf_meta['path_in_repo']}")
